In [1]:
print(0)

0


In [2]:
import os
import pandas as pd

SCORES_ROOT = "/home/pratyushprasad/ACL/SCORES_1"

EXPECTED_FILES = [
    "allminilm.csv",
    "sapbert.csv",
    "p_bert.csv",   "r_bert.csv",   "f_bert.csv",
    "p_biomedbert.csv", "r_biomedbert.csv", "f_biomedbert.csv",
    "p_bioclinbert.csv","r_bioclinbert.csv","f_bioclinbert.csv",
    "gemma_4b.csv",
    "llama_8b.csv",
]

SPLITS = ["DWTD", "OSD", "RPD"]

for split in SPLITS:
    split_path = os.path.join(SCORES_ROOT, split)
    if not os.path.isdir(split_path):
        print(f"\n{'='*60}")
        print(f"  {split}  — directory not found")
        continue

    models = sorted(os.listdir(split_path))
    models = [m for m in models if os.path.isdir(os.path.join(split_path, m))]

    rows = []
    for model in models:
        model_path = os.path.join(split_path, model)
        row = {"model": model}
        for f in EXPECTED_FILES:
            fpath = os.path.join(model_path, f)
            if os.path.exists(fpath):
                try:
                    n = len(pd.read_csv(fpath))
                    row[f] = n
                except Exception:
                    row[f] = "ERR"
            else:
                row[f] = "MISSING"
        rows.append(row)

    df = pd.DataFrame(rows).set_index("model")

    print(f"\n{'='*60}")
    print(f"  {split}")
    print('='*60)
    print(df.to_string())
    print()


  DWTD
                allminilm.csv sapbert.csv p_bert.csv r_bert.csv f_bert.csv p_biomedbert.csv r_biomedbert.csv f_biomedbert.csv p_bioclinbert.csv r_bioclinbert.csv f_bioclinbert.csv gemma_4b.csv llama_8b.csv
model                                                                                                                                                                                                        
dermatollama              530         530        530        530        530              530              530              530               530               530               530      MISSING      MISSING
gemini                    494         494        494        494        494              494              494              494               494               494               494      MISSING      MISSING
lingshu_32b               530         530        530        530        530              530              530              530               530               530       

In [3]:
import os
import pandas as pd

SCORES_ROOT = "/home/pratyushprasad/ACL/SCORES_1"

ALL_COLS = [
    "type_of_lesion", "site", "count", "arrangement", "size",
    "color_pattern", "border", "surface_changes",
    "presence_of_exudate_or_discharge", "surrounding_skin_changes",
    "secondary_changes", "pattern_or_shape", "differential_diagnosis",
    "additional_notes", "overall_description",
]

rows = []

for split in sorted(os.listdir(SCORES_ROOT)):
    split_path = os.path.join(SCORES_ROOT, split)
    if not os.path.isdir(split_path):
        continue
    for model in sorted(os.listdir(split_path)):
        model_path = os.path.join(split_path, model)
        fpath = os.path.join(model_path, "sapbert.csv")
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath, dtype=str)
        # coerce feature cols to numeric, missing/invalid → NaN
        for c in ALL_COLS:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            else:
                df[c] = float("nan")
        means = df[ALL_COLS].mean(skipna=True)
        row = {"split/model": f"{split}/{model}"}
        for c in ALL_COLS:
            row[c] = round(means[c], 3) if not pd.isna(means[c]) else "—"
        rows.append(row)

result = pd.DataFrame(rows).set_index("split/model")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.float_format", "{:.3f}".format)
print(result.to_string())

                      type_of_lesion  site  count  arrangement  size  color_pattern  border  surface_changes  presence_of_exudate_or_discharge  surrounding_skin_changes  secondary_changes  pattern_or_shape  differential_diagnosis  additional_notes  overall_description
split/model                                                                                                                                                                                                                                                                 
DWTD/dermatollama              0.524 0.543  0.587        0.558 0.588          0.654   0.633            0.553                             0.589                     0.600              0.523             0.589                   0.629             0.540                0.743
DWTD/gemini                    0.694 0.634  0.631        0.559 0.590          0.689   0.528            0.573                             0.544                     0.585              0.553      

In [ ]:
.khjbkjhkjhv

In [ ]:
import os
import pandas as pd

SCORES_ROOT = "/home/pratyushprasad/ACL/SCORES"

ALL_COLS = [
    "type_of_lesion", "site", "count", "arrangement", "size",
    "color_pattern", "border", "surface_changes",
    "presence_of_exudate_or_discharge", "surrounding_skin_changes",
    "secondary_changes", "pattern_or_shape", "differential_diagnosis",
    "additional_notes", "overall_description",
]

rows = []

split_path = os.path.join(SCORES_ROOT, "RPD")
for model in sorted(os.listdir(split_path)):
    model_path = os.path.join(split_path, model)
    fpath = os.path.join(model_path, "sapbert.csv")
    if not os.path.exists(fpath):
        continue
    df = pd.read_csv(fpath, dtype=str)
    for c in ALL_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = float("nan")
    means = df[ALL_COLS].mean(skipna=True)

    row = {"model": model}
    numeric_vals = []
    for c in ALL_COLS:
        v = means[c]
        if pd.isna(v):
            row[c] = "—"
            numeric_vals.append(0.0)   # missing counts as 0 in average
        else:
            row[c] = round(v, 3)
            numeric_vals.append(v)
    row["average_score"] = round(sum(numeric_vals) / len(ALL_COLS), 3)
    rows.append(row)

result = pd.DataFrame(rows).set_index("model")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 400)
print(result.to_string())

In [ ]:
import os
import pandas as pd

SCORES_ROOT = "/home/pratyushprasad/ACL/SCORES"

ALL_COLS = [
    "type_of_lesion", "site", "count", "arrangement", "size",
    "color_pattern", "border", "surface_changes",
    "presence_of_exudate_or_discharge", "surrounding_skin_changes",
    "secondary_changes", "pattern_or_shape", "differential_diagnosis",
    "additional_notes", "overall_description",
]

rows = []

split_path = os.path.join(SCORES_ROOT, "RPD")
for model in sorted(os.listdir(split_path)):
    model_path = os.path.join(split_path, model)
    fpath = os.path.join(model_path, "gemma_4b.csv")
    if not os.path.exists(fpath):
        continue
    df = pd.read_csv(fpath, dtype=str)
    for c in ALL_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = float("nan")

    means = df[ALL_COLS].mean(skipna=True)

    row = {"model": model}
    numeric_vals = []
    for c in ALL_COLS:
        v = means[c]
        if pd.isna(v):
            row[c] = "—"
            numeric_vals.append(0.0)
        else:
            row[c] = round(v, 3)
            numeric_vals.append(v)

    row["average_score"] = round(sum(numeric_vals) / len(ALL_COLS), 3)
    row["count_10"] = int((df[ALL_COLS] == 10).sum().sum())
    rows.append(row)

result = pd.DataFrame(rows).set_index("model")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 400)
print(result.to_string())

In [ ]:
# result.to_csv("/home/pratyushprasad/ACL/SCORES/scores_3d/OSD_gemma_4.csv", index=True)

In [ ]:
import os
import pandas as pd

SCORES_ROOT = "/home/pratyushprasad/ACL/SCORES"
OUT_ROOT    = "/home/pratyushprasad/ACL/SCORES/scores_3d"
os.makedirs(OUT_ROOT, exist_ok=True)

ALL_COLS = [
    "type_of_lesion", "site", "count", "arrangement", "size",
    "color_pattern", "border", "surface_changes",
    "presence_of_exudate_or_discharge", "surrounding_skin_changes",
    "secondary_changes", "pattern_or_shape", "differential_diagnosis",
    "additional_notes", "overall_description",
]

METRICS = [
    "allminilm", "sapbert",
    "p_bert",    "r_bert",    "f_bert",
    "p_biomedbert", "r_biomedbert", "f_biomedbert",
    "p_bioclinbert","r_bioclinbert","f_bioclinbert",
    "gemma_4b", "llama_8b",
]

SPLITS = ["RPD", "OSD", "DWTD"]

# Expected row counts per split (for sanity check printout)
EXPECTED = {"RPD": 2118, "OSD": 250, "DWTD": 530}


def aggregate(split, metric):
    split_path = os.path.join(SCORES_ROOT, split)
    if not os.path.isdir(split_path):
        return None

    rows = []
    for model in sorted(os.listdir(split_path)):
        model_path = os.path.join(split_path, model)
        if not os.path.isdir(model_path):
            continue
        fpath = os.path.join(model_path, f"{metric}.csv")
        if not os.path.exists(fpath):
            continue

        df = pd.read_csv(fpath, dtype=str)
        for c in ALL_COLS:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            else:
                df[c] = float("nan")

        total_rows = len(df)
        means = df[ALL_COLS].mean(skipna=True)

        row = {"model": model, "row_count": total_rows}
        numeric_vals = []
        for c in ALL_COLS:
            v = means[c]
            if pd.isna(v):
                row[c] = float("nan")
                numeric_vals.append(0.0)
            else:
                row[c] = round(v, 4)
                numeric_vals.append(v)
        row["average_score"] = round(sum(numeric_vals) / len(ALL_COLS), 4)
        rows.append(row)

    if not rows:
        return None

    cols = ["model", "row_count"] + ALL_COLS + ["average_score"]
    return pd.DataFrame(rows, columns=cols).set_index("model")


# ── main ──────────────────────────────────────────────────────────────────────
for metric in METRICS:
    for split in SPLITS:
        result = aggregate(split, metric)
        out_path = os.path.join(OUT_ROOT, f"{metric}_{split}.csv")

        if result is None:
            print(f"  SKIP  {metric}_{split}.csv  (no data)")
            continue

        result.to_csv(out_path)

        # sanity: check row_count values match expected total
        counts = result["row_count"].tolist()
        ok = all(c == EXPECTED[split] for c in counts)
        flag = "✓" if ok else f"⚠  expected {EXPECTED[split]}, got {counts}"
        print(f"  {metric}_{split}.csv  —  {len(result)} models  {flag}")

print(f"\nDone. Files saved to {OUT_ROOT}")

In [ ]:
"""
For DWTD split, llama_8b.csv only:
- Split by dataset_name (5 unique values)
- For each dataset: compute per-model feature averages + average_score
- Save one CSV per dataset to /home/pratyushprasad/ACL/SCORES/scores_llama_8b/
"""

import os
import pandas as pd

SCORES_ROOT = "/home/pratyushprasad/ACL/SCORES"
OUT_ROOT    = "/home/pratyushprasad/ACL/SCORES/scores_gemma_4b"
os.makedirs(OUT_ROOT, exist_ok=True)

ALL_COLS = [
    "type_of_lesion", "site", "count", "arrangement", "size",
    "color_pattern", "border", "surface_changes",
    "presence_of_exudate_or_discharge", "surrounding_skin_changes",
    "secondary_changes", "pattern_or_shape", "differential_diagnosis",
    "additional_notes", "overall_description",
]

dwtd_path = os.path.join(SCORES_ROOT, "DWTD")

for model in sorted(os.listdir(dwtd_path)):
    model_path = os.path.join(dwtd_path, model)
    if not os.path.isdir(model_path):
        continue
    fpath = os.path.join(model_path, "gemma_4b.csv")
    if not os.path.exists(fpath):
        continue

    df = pd.read_csv(fpath, dtype=str)

    if "dataset_name" not in df.columns:
        print(f"  WARN: no dataset_name column in {fpath}")
        continue

    # coerce feature cols to numeric
    for c in ALL_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = float("nan")

    datasets = sorted(df["dataset_name"].dropna().unique())
    print(f"\n  {model} — datasets: {datasets}")

    for dname in datasets:
        sub = df[df["dataset_name"] == dname]
        sub = sub.dropna(subset=ALL_COLS, how="all")   # ← drop rows where every feature is NaN
        means = sub[ALL_COLS].mean(skipna=True)
        numeric_vals = []
        row = {}
        for c in ALL_COLS:
            v = means[c]
            if pd.isna(v):
                row[c] = float("nan")
                numeric_vals.append(0.0)
            else:
                row[c] = round(v, 4)
                numeric_vals.append(v)
        row["average_score"] = round(sum(numeric_vals) / len(ALL_COLS), 4)
        row["row_count"] = len(sub)

        # safe filename from dataset name
        safe_name = dname.strip().replace(" ", "_").replace("/", "-")
        out_path = os.path.join(OUT_ROOT, f"{safe_name}.csv")

        # append this model as a row
        result_row = {"model": model, "row_count": row["row_count"]}
        result_row.update({c: row[c] for c in ALL_COLS})
        result_row["average_score"] = row["average_score"]

        # read existing if already written by a previous model loop
        if os.path.exists(out_path):
            existing = pd.read_csv(out_path)
            updated = pd.concat([existing, pd.DataFrame([result_row])], ignore_index=True)
        else:
            updated = pd.DataFrame([result_row])

        updated.to_csv(out_path, index=False)
        print(f"    → {safe_name}.csv  rows_in_dataset={row['row_count']}  avg={row['average_score']}")

print(f"\nDone. Files saved to {OUT_ROOT}")